| título | projeto | versão | data | autores | status |
| :--- | :--- | :--- | :--- | :--- | :--- |
| CRISP-DM — Fase 3: Data Preparation | Projeção da Taxa de Congestionamento — Justiça Estadual (GO) | 1.0 | 01-12-2025 | Júlio César e Lays de Freitas | Rascunho |

Esse Notebook contém o **pré-processamento dos dados**.

### BIBLIOTECAS

In [13]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import glob

from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from datetime import date

### CARREGAMENTO DOS DADOS

In [14]:
# Listar os arquivos CSV na pasta 'uploads'
arquivos_csv = glob.glob('uploads/processos_*.csv')

# Carregar os arquivos CSV e concatenar em um único DataFrame
dfs = []
for arquivo in arquivos_csv:
    
    df_temp = pd.read_csv(arquivo, sep=',', encoding='utf-8')
    dfs.append(df_temp)

dataset = pd.concat(dfs, ignore_index=True)

print("\n=== Arquivo carregado com sucesso! ===")
print("Dimensões (linhas, colunas):", dataset.shape)


=== Arquivo carregado com sucesso! ===
Dimensões (linhas, colunas): (3245632, 10)


### GRAVANDO UMA CÓPIA PARA TRABALHO

In [15]:
df = dataset.copy()

### AMOSTRA DOS DADOS

In [16]:
df.head()

,processo,data_distribuicao,data_baixa,entrancia,comarca,serventia,nome_area_acao,is_segredo_justica,codg_classe,codg_assuntos
0,0119071.75.2004.8.09.0051,2022-05-25,2022-06-30,FINAL,GOIÂNIA,2ª Vara Cível,upj civel,False,7.0,10671
1,0168391.94.2004.8.09.0051,2022-05-20,2022-05-20,FINAL,GOIÂNIA,3ª Vara Cível,upj civel,False,7.0,10671
2,0189657.40.2004.8.09.0051,2022-06-02,2024-01-22,FINAL,GOIÂNIA,31ª Vara Cível,upj civel,False,7.0,10671
3,0197944.89.2004.8.09.0051,2022-06-07,2022-10-07,FINAL,GOIÂNIA,22ª Vara Cível,upj civel,False,7.0,10671
4,0211274.56.2004.8.09.0051,2022-06-09,2022-08-03,FINAL,GOIÂNIA,8ª Vara Cível,upj civel,False,7.0,10671


### LIMPEZA E TRATAMENTO DOS DADOS

In [17]:
# Verificar o nome correto das colunas (pode haver diferenças de acentuação ou espaços)
colunas = df.columns.tolist()

# Encontrar as colunas de data corretamente
coluna_serventia = [col for col in colunas if 'serventia' in col.lower()][0]
coluna_distribuicao = [col for col in colunas if 'data_distribuicao' in col.lower()][0]
coluna_baixa = [col for col in colunas if 'data_baixa' in col.lower()][0]
coluna_area_acao = [col for col in colunas if 'nome_area_acao' in col.lower()][0]
coluna_processo_id = [col for col in colunas if 'processo' in col.lower()][0]
coluna_comarca = [col for col in colunas if 'comarca' in col.lower()][0]

# Renomear colunas para garantir consistência
df = df.rename(columns={
coluna_distribuicao: 'data_distribuicao',
coluna_baixa: 'data_baixa',
coluna_area_acao: 'nome_area_acao',
coluna_processo_id: 'processo',
coluna_comarca: 'comarca',
coluna_serventia: 'serventia'
})

# Converter colunas de data para datetime com tratamento de erros
df['data_distribuicao'] = pd.to_datetime(df['data_distribuicao'], errors='coerce')
df['data_baixa'] = pd.to_datetime(df['data_baixa'], errors='coerce')

### CONSTRUÇÃO DO DATAFRAME DE TREINO E TESTE

In [18]:
# CRIAÇÃO DAS ESTATÍSTICAS POR MÊS ('comarca' e 'serventia') >> Revisado
# --- 1. PREPARAÇÃO DOS DADOS ---
# Extração de componentes de data
print("Extraindo datas...")
df['ano_distribuicao'] = df['data_distribuicao'].dt.year
df['mes_distribuicao'] = df['data_distribuicao'].dt.month
df['dia_distribuicao'] = df['data_distribuicao'].dt.day

df['ano_baixa'] = df['data_baixa'].dt.year
df['mes_baixa'] = df['data_baixa'].dt.month
df['dia_baixa'] = df['data_baixa'].dt.day

# Chaves de agrupamento
grouping_keys = ['comarca', 'serventia']

# ==============================================================================
# FUNÇÃO GENÉRICA DE CÁLCULO (Para evitar repetição de código)
# ==============================================================================
def calcular_estatisticas_cohort(df_main, cols_dist, cols_baixa, nome_periodo):
    """
    Calcula Distribuídos, Baixados e Pendentes.
    REVISÃO DOS CÁLCULOS:
      - Distribuídos: contagem por data_distribuicao (entrada)
      - Baixados: contagem por data_baixa (referência), dentro do par entrada->referência
      - Pendentes: contagem de data_baixa nula/vazia (NaT), agrupada por entrada
    """
    
    # 1. Calcular TOTAL DE DISTRIBUÍDOS
    cols_group_dist = cols_dist + grouping_keys
    df_dist = df_main.groupby(cols_group_dist)['processo'].nunique().reset_index(name=f'Distribuídos{nome_periodo}')
    
    # 2. Calcular TOTAL DE BAIXADOS (somente registros com baixa)
    cols_group_baixa = cols_dist + cols_baixa + grouping_keys
    df_baixa = df_main.dropna(subset=cols_baixa).groupby(cols_group_baixa)['processo'].nunique().reset_index(name=f'Baixados{nome_periodo}')
    
    # 3. Calcular TOTAL DE PENDENTES (data_baixa nula/vazia -> componentes de baixa NaN)
    df_pend = df_main[df_main[cols_baixa[0]].isna()].groupby(cols_group_dist)['processo'].nunique().reset_index(name=f'Pendentes{nome_periodo}')
    
    # 4. CRIAÇÃO DO GRID (Cross Join)
    unique_dist = df_dist[cols_dist].drop_duplicates()
    unique_baixa = df_main[cols_baixa].dropna().drop_duplicates()
    unique_units = df_main[grouping_keys].drop_duplicates()
    
    # Cross Join 1: Datas de Dist x Datas de Baixa (usando merge dummy para performance)
    df_dates = pd.merge(
        unique_dist.assign(key=1), 
        unique_baixa.assign(key=1), 
        on='key'
    ).drop('key', axis=1)
    
    # --- FILTRO DE DATAS ---
    if len(cols_dist) == 1: # Anual
        df_dates = df_dates[df_dates[cols_baixa[0]] >= df_dates[cols_dist[0]]]
        
    elif len(cols_dist) == 2: # Mensal
        ano_d = df_dates[cols_dist[0]].astype(int).astype(str)
        mes_d = df_dates[cols_dist[1]].astype(int).astype(str)
        
        ano_b = df_dates[cols_baixa[0]].astype(int).astype(str)
        mes_b = df_dates[cols_baixa[1]].astype(int).astype(str)
        
        d_dist = pd.to_datetime(ano_d + '-' + mes_d + '-01')
        d_baixa = pd.to_datetime(ano_b + '-' + mes_b + '-01')
        
        df_dates = df_dates[d_baixa >= d_dist]

    # Cross Join 2: (Datas) x (Comarca/Serventia)
    df_grid = pd.merge(
        df_dates.assign(key=1),
        unique_units.assign(key=1),
        on='key'
    ).drop('key', axis=1)
    
    # 5. MERGES (Juntar dados reais no Grid)
    df_final = pd.merge(df_grid, df_dist, on=cols_dist + grouping_keys, how='left')
    df_final = pd.merge(df_final, df_baixa, on=cols_dist + cols_baixa + grouping_keys, how='left')
    df_final = pd.merge(df_final, df_pend, on=cols_dist + grouping_keys, how='left')
    
    # Preencher Zeros
    df_final[f'Distribuídos{nome_periodo}'] = df_final[f'Distribuídos{nome_periodo}'].fillna(0).astype(int)
    df_final[f'Baixados{nome_periodo}'] = df_final[f'Baixados{nome_periodo}'].fillna(0).astype(int)
    df_final[f'Pendentes{nome_periodo}'] = df_final[f'Pendentes{nome_periodo}'].fillna(0).astype(int)
    
    # Filtrar apenas onde houve distribuição
    df_final = df_final[df_final[f'Distribuídos{nome_periodo}'] > 0].copy()
    
    # 6. TAXA DE CONGESTIONAMENTO (com a definição solicitada)
    soma = df_final[f'Baixados{nome_periodo}'] + df_final[f'Pendentes{nome_periodo}']
    df_final[f'Taxa de Congestionamento{nome_periodo} (%)'] = np.where(
        soma > 0, (df_final[f'Pendentes{nome_periodo}'] / soma) * 100, 0
    ).round(2)

    # 7. CONVERSÃO FINAL PARA INTEIRO (NOVO BLOCO)
    cols_tempo = cols_dist + cols_baixa
    for col in cols_tempo:
        if col in df_final.columns:
            df_final[col] = df_final[col].astype(int)

    return df_final

# ==============================================================================
# 2. MONTANDO O DATASET COM OS CÁLCULOS MENSAIS
# ==============================================================================
print("Calculando estatísticas mensais...")
df_estatisticas = calcular_estatisticas_cohort(
    df, 
    cols_dist=['ano_distribuicao', 'mes_distribuicao'], 
    cols_baixa=['ano_baixa', 'mes_baixa'], 
    nome_periodo='_mes'
)

# Exclusão de Colunas Desnecessárias
cols_to_drop = [
    'ano_distribuicao',    
    'mes_distribuicao', 
    'ano_baixa',       
    'mes_baixa'                   
]

# Dropamos apenas o que existe no dataframe
df2 = df_estatisticas.drop(columns=[c for c in cols_to_drop if c in df.columns], errors='ignore')

df3 = df2.copy()

# Inclusão da coluna mês de referência para Aplicação de ML
df3["mes_ref"] = pd.to_datetime(
    df_estatisticas["ano_baixa"].astype(str) + "-" + df_estatisticas["mes_baixa"].astype(str).str.zfill(2) + "-01"
)

# Dataset pronto
df_estatisticas_mes = df3.sort_values("mes_ref")

print("Concluído!")

Extraindo datas...
Calculando estatísticas mensais...
Concluído!


### AMOSTRA DO DATAFRAME TRATADO

In [19]:
# CRIAÇÃO DAS ESTATÍSTICAS POR MÊS ('comarca' e 'serventia'):

# 1. Extrair MÊS e ANO das colunas data_distribuicao e data_baixa
df['mes_distribuicao'] = df['data_distribuicao'].dt.month
df['mes_baixa'] = df['data_baixa'].dt.month
df['ano_distribuicao'] = df['data_distribuicao'].dt.year
df['ano_baixa'] = df['data_baixa'].dt.year

# Chave de agrupamento para as estatísticas
grouping_keys = ['comarca', 'serventia']

# 2. Cálculos MENSAIS
# 2.1 Distribuídos por MÊS
distribuidos_mes_df = df.dropna(subset=['ano_distribuicao', 'mes_distribuicao']).groupby(
    ['ano_distribuicao', 'mes_distribuicao'] + grouping_keys
).size().reset_index(name='Distribuídos_mes')
distribuidos_mes_df = distribuidos_mes_df.rename(columns={'ano_distribuicao': 'ano', 'mes_distribuicao': 'mes'})

# 2.2 Baixados por MÊS
baixados_mes_df = df.dropna(subset=['ano_baixa', 'mes_baixa']).groupby(
    ['ano_baixa', 'mes_baixa'] + grouping_keys
).size().reset_index(name='Baixados_mes')
baixados_mes_df = baixados_mes_df.rename(columns={'ano_baixa': 'ano', 'mes_baixa': 'mes'})

# 2.3 Pendentes por MÊS
pendentes_mes_df = df[df['data_baixa'].isna()].dropna(subset=['ano_distribuicao', 'mes_distribuicao']).groupby(
    ['ano_distribuicao', 'mes_distribuicao'] + grouping_keys
).size().reset_index(name='Pendentes_mes')
pendentes_mes_df = pendentes_mes_df.rename(columns={'ano_distribuicao': 'ano', 'mes_distribuicao': 'mes'})

# 4. Junção e Limpeza (MENSAL)
merge_keys_mes = ['ano', 'mes'] + grouping_keys
estatisticas_mes = pd.merge(distribuidos_mes_df, baixados_mes_df, on=merge_keys_mes, how='outer')
estatisticas_mes = pd.merge(estatisticas_mes, pendentes_mes_df, on=merge_keys_mes, how='outer')
estatisticas_mes = estatisticas_mes.fillna(0)
estatisticas_mes[['Distribuídos_mes', 'Baixados_mes', 'Pendentes_mes']] = estatisticas_mes[['Distribuídos_mes', 'Baixados_mes', 'Pendentes_mes']].astype(int)
estatisticas_mes = estatisticas_mes.dropna(subset=['ano', 'mes'])
estatisticas_mes[['ano', 'mes']] = estatisticas_mes[['ano', 'mes']].astype(int)

# 5. Cálculo da Taxa de Congestionamento (MENSAL)
soma_mensal = estatisticas_mes['Pendentes_mes'] + estatisticas_mes['Baixados_mes']
estatisticas_mes['Taxa de Congestionamento_mes (%)'] = np.where(
    soma_mensal > 0, (estatisticas_mes['Pendentes_mes'] / soma_mensal) * 100, 0
).round(2)

# 6. Montando o dataframe
estatisticas_mes = estatisticas_mes.sort_values(by=['ano', 'mes', 'comarca', 'serventia'], ascending=[False, True, True, True])
colunas_finais_mes = [
    'ano', 'mes', 'comarca', 'serventia',
    'Distribuídos_mes', 'Baixados_mes', 'Pendentes_mes', 'Taxa de Congestionamento_mes (%)'
]

df_estatisticas_mes = estatisticas_mes[colunas_finais_mes]

# 7. Amostra do dataframe tratado
df_estatisticas_mes.head()


,ano,mes,comarca,serventia,Distribuídos_mes,Baixados_mes,Pendentes_mes,Taxa de Congestionamento_mes (%)
19253,2025,1,ABADIÂNIA,Vara Judicial,140,66,80,54.79
19254,2025,1,ACREÚNA,"1ª Vara Judicial (Família e Sucessões, Infânci...",109,109,43,28.29
19255,2025,1,ACREÚNA,"2ª Vara Judicial (Fazendas Públicas, Criminal,...",67,84,41,32.80
19256,2025,1,ALEXÂNIA,Vara Judicial,258,225,99,30.56
19257,2025,1,ALTO PARAÍSO DE GOIÁS,Vara Judicial,155,70,96,57.83


### SEPARAR CONJUNTOS DE FORMA TEMPORAL DE TREINO(tudo antes dos últimos 3 meses) E TESTE (últimos 3 meses)

In [ ]:

# Configurações visuais
plt.style.use('ggplot')
pd.set_option('display.max_columns', None)

df_full = df_estatisticas_mes.copy()

# Garantir que a coluna de data é datetime
df_full['mes_ref'] = pd.to_datetime(df_full['mes_ref'])
#df_full['Taxa de Congestionamento_mes'] = df_full['Taxa de Congestionamento_mes (%)'] * 100  # Converter para porcentagem

# Ordenar por unidade e data
df_full = df_full.sort_values(by=['comarca', 'serventia', 'mes_ref'])

print(f"Total de registros carregados: {len(df_full)}")
print(f"Período dos dados: de {df_full['mes_ref'].min().date()} até {df_full['mes_ref'].max().date()}")

### AMOSTRA DO CONJUNTO TREINO

In [21]:
train.head()

comarca      serventia  Distribuídos_mes  Baixados_mes  \
155710     GOIANÁPOLIS  Vara Judicial               136             6   
593690        ANÁPOLIS  5ª Vara Cível               220             0   
275501      CORUMBAÍBA  Vara Judicial                86             4   
440545  CACHOEIRA ALTA  Vara Judicial                92            11   
515389        JOVIÂNIA  Vara Judicial                65             1   

        Pendentes_mes  Taxa de Congestionamento_mes (%)    mes_ref  
155710             33                             84.62 2023-03-01  
593690            163                            100.00 2025-10-01  
275501             11                             73.33 2025-06-01  
440545             14                             56.00 2024-01-01  
515389             15                             93.75 2025-05-01

### GRAVAR O CONJUNTO TREINO PRÉ-PROCESSADO

In [22]:
train.to_csv('datasets/train-processed.csv', index=False)

### AMOSTRA DO CONJUNTO TESTE

In [23]:
test_split.head()

comarca                                          serventia  \
474019    GOIÂNIA                                     10ª Vara Cível   
251068    FORMOSA                3ª Vara Cível e Família e Sucessões   
149237   MINEIROS                 Vara de Família, Sucessões e Cível   
245193    GOIÂNIA  5ª Vara da Fazenda Pública Municipal e de Regi...   
74772   ITUMBIARA               1º Juizado Especial Cível e Criminal   

        Distribuídos_mes  Baixados_mes  Pendentes_mes  \
474019               116             3             39   
251068                90             2             21   
149237                93             1             25   
245193               173             1             96   
74772                178             0              0   

        Taxa de Congestionamento_mes (%)    mes_ref  
474019                             92.86 2024-03-01  
251068                             91.30 2023-04-01  
149237                             96.15 2025-01-01  
245193                             98.97 2024-08-01  
74772                               0.00 2024-07-01

### GRAVAR CONJUNTO TESTE PRÉ-PROCESSADO

In [24]:
test_split.to_csv('datasets/test_split.csv', index=False)